# Demanda semanal observada y población común
El RAW permanece intacto. La copia analítica elimina transacciones con `cantidad` nula o negativa **antes** de agregar; conserva ceros y no convierte cajas/unidades. El objetivo es cantidad positiva observada en ERP (incluyendo semanas de cero), no saldo neto ni demanda latente.

Agregación `W-SUN`, ceros sólo en huecos interiores del calendario depurado de cada serie. No se extienden los extremos sin evidencia. Corte `2026-01-01`; se excluye del scoring la semana 29-dic a 4-ene. Las semanas de los extremos pueden ser parciales. Las medias móviles usan `shift(1)` y el calendario completo se conserva al calcular lags.

La tabla final se calcula desde la intersección exacta `sucursal + producto + semana` de las predicciones válidas de los cuatro modelos. Las métricas propias quedan separadas. Precisión completa en archivos y cálculos; cuatro decimales sólo en pantalla.

ARIMA conserva ADF y búsqueda AIC p/q=0..3, con mínimos 20/10/2 semanas. Registra el estado y motivo de cada serie. Se pronostica también la semana cruzada, sin puntuarla ni comprimir el horizonte. Su protocolo sigue siendo multi-step desde origen fijo; compartir población con ML no iguala la información disponible en cada pronóstico.


In [1]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path('..').resolve()))
from src.experimento_semanal import SALIDA
pd.options.display.float_format = '{:.4f}'.format
from src.experimento_semanal import ejecutar_arima
arima_model, metricas_arima = ejecutar_arima()
display(pd.DataFrame([metricas_arima]))
display(pd.DataFrame([arima_model.conteo_series]))
display(arima_model.registro_series.groupby(['estado','motivo'], dropna=False).size().rename('series').reset_index())
print('Ejecutar 07 para construir la comparación común; estas métricas son de cobertura propia.')


Iniciando entrenamiento ARIMA (Iteracion por Sucursal y Producto)...


Semanas excluidas por cruce del corte: 144
Resumen de series ARIMA: {'potenciales': 266, 'encontradas': 240, 'descartadas': 111, 'fallidas': 0, 'modeladas': 129}
Entrenamiento completado.


,Modelo,MAE,RMSE,R2,Observaciones_evaluadas,Series_evaluadas
0,ARIMA (Optimizado ADF/AIC),0.2363,0.5797,0.2175,3444,129


,potenciales,encontradas,descartadas,fallidas,modeladas
0,266,240,111,0,129


,estado,motivo,series
0,descartada,menos_de_10_semanas_train,22
1,descartada,menos_de_20_semanas,17
2,descartada,menos_de_2_semanas_test,72
3,modelada,,129


Ejecutar 07 para construir la comparación común; estas métricas son de cobertura propia.
